# Criterium Research API Tutorial
This notebook will walk you through interacting with the FastAPI application we just built. 

**Prerequisites:**
1. Ensure your server is running (`make run` in a terminal).
2. Ensure your `.env` file contains valid API keys for Gemini and Firecrawl.

In [1]:
import requests
import json

BASE_URL = "http://127.0.0.1:8000"

def print_json(data):
    print(json.dumps(data, indent=2))

## 1. Create a Research Collection
Since we are using an in-memory SQLite database (`:memory:`), each server restart begins with an empty database. Let's create our test collection first using the Book Series schema.

Notice that because we share the `schemas` between the client and server, we can build the schema using pure Pydantic and never write raw JSON manually!

In [ ]:
from criterium import schemas

schema = schemas.ResearchSchemaObject(
  properties={
    "title": schemas.ResearchSchemaString(
      description="The exact title of the primary book."
    ),
    "year": schemas.ResearchSchemaInteger(
      description="The 4-digit year the book was published."
    ),
    "series_books": schemas.ResearchSchemaArray(
      description="Other books in the same series.",
      items=schemas.ResearchSchemaObject(
        properties={
          "title": schemas.ResearchSchemaString(),
          "chronological_order": schemas.ResearchSchemaInteger()
        },
        required=["title"]
      )
    )
  },
  required=["title", "year"]
)

# Use our strictly typed Pydantic models to create the payload!
payload = schemas.CollectionCreate(
    name="Book Series Researcher",
    extraction_prompt="You are a literary researcher. Extract the details of the primary book discussed on this page, as well as a list of any other books mentioned that belong to the same series.",
    research_schema=schema
)

try:
    response = requests.post(f"{BASE_URL}/collections", json=payload.model_dump(exclude_none=True))
    response.raise_for_status()
    
    # We can validate the response straight back into our Pydantic model
    collection_data = schemas.CollectionResponse.model_validate(response.json())
    collection_id = collection_data.id
    print(f"Created collection with ID: {collection_id}")
    print_json(collection_data.model_dump())
    
except requests.exceptions.ConnectionError:
    print("Error: Could not connect to the server at", BASE_URL)
    print("Please make sure your server is running in a terminal with: make run")
except Exception as e:
    print("Request failed:", e)

Status Code: 200
Response text: {"id":2,"name":"Book Series Researcher","extraction_prompt":"You are a literary researcher. Extract the details of the primary book discussed on this page, as well as a list of any other books mentioned that belong to the same series.","research_schema":{"type":"object","description":null,"properties":{"title":{"type":"string","description":"The exact title of the primary book.","properties":null,"items":null,"required":null},"year":{"type":"integer","description":"The 4-digit year the book was published.","properties":null,"items":null,"required":null},"series_books":{"type":"array","description":"Other books in the same series.","properties":null,"items":{"type":"object","description":null,"properties":{"title":{"type":"string","description":null,"properties":null,"items":null,"required":null},"chronological_order":{"type":"integer","description":null,"properties":null,"items":null,"required":null}},"items":null,"required":["title"]},"required":null}},

## 2. Verify / List All Collections
Now let's query `GET /collections` to verify our newly created collection is listed in the database.

In [ ]:
response = requests.get(f"{BASE_URL}/collections")

# Parse list of Pydantic models
collections = [schemas.CollectionResponse.model_validate(c) for c in response.json()]

print(f"Found {len(collections)} collections.")
for col in collections:
    print(f"- {col.id}: {col.name}")

Status Code: 200
[
  {
    "id": 1,
    "name": "Book Series Researcher",
    "extraction_prompt": "You are a literary researcher. Extract the details of the primary book discussed on this page, as well as a list of any other books mentioned that belong to the same series.",
    "research_schema": {
      "type": "object",
      "description": null,
      "properties": {
        "title": {
          "type": "string",
          "description": "The exact title of the primary book.",
          "properties": null,
          "items": null,
          "required": null
        },
        "year": {
          "type": "integer",
          "description": "The 4-digit year the book was published.",
          "properties": null,
          "items": null,
          "required": null
        },
        "series_books": {
          "type": "array",
          "description": "Other books in the same series.",
          "properties": null,
          "items": {
            "type": "object",
            "descr

## 3. Extract a Product (The Research Protocol)
Now we'll trigger the 4-stage research pipeline. We'll pass a search query to our endpoint.
Behind the scenes, this will:
1. Search and Scrape using Firecrawl
2. Save the raw markdown to SQLite
3. Run the markdown against Gemini using our JSON Schema
4. Save the structured JSON back to SQLite

In [ ]:
# Note: This will fail if your .env keys are still set to "placeholder"
query_payload = schemas.ProductCreate(
    product_info="The Lord of the Rings: The Fellowship of the Ring book"
)

# This request might take a few seconds as it hits both APIs
print(f"Triggering extraction for collection {collection_id}...")
extract_response = requests.post(f"{BASE_URL}/collections/{collection_id}/products", json=query_payload.model_dump())

print("Status Code:", extract_response.status_code)
print_json(extract_response.json())

Triggering extraction for collection 2...
Status Code: 500
{
  "detail": "Discovery failed: FirecrawlClient.search() got an unexpected keyword argument 'scrapeOptions'. Did you mean 'scrape_options'?"
}


## 4. View the Extracted Data
Finally, let's pull down the products we just researched to see the raw markdown and the structured JSON output!

In [ ]:
products_response = requests.get(f"{BASE_URL}/collections/{collection_id}/products")

# We can parse the response straight into our Pydantic List
products = [schemas.ProductResponse.model_validate(p) for p in products_response.json()]

for product in products:
    print("="*50)
    print("PRODUCT:", product.name)
    print("URL:", product.source_url)
    print("-" * 50)
    print("EXTRACTED DATA:")
    print_json(product.extracted_data)
    print("="*50)